# UVA → LIBERO-10 multi-task 평가 (Colab)

UVA 논문(arXiv:2503.00200v3) **Table I**의 LIBERO-10 컬럼을 재현(혹은 근사)합니다.
논문 수치: **UVA = 0.90** (10개 태스크, 각각 50 rollout 평균).
Supplementary Table VIII: **UVA = 0.93** (30-test 세팅, 태스크당 3 rollout).

## 이 노트북이 하는 일
1. LIBERO + robosuite + robomimic + UVA 의존성 설치.
2. 공개 체크포인트 `libero10.ckpt`와 LIBERO-10 hdf5 다운로드.
3. `eval_sim.py`와 동일한 평가 로직을 인라인으로 실행 — `n_test`를 열어둔 **3단계 예산**:
   - **Smoke** (T4 기준 약 10–20분): 1 태스크, 1 rollout. 파이프라인 산도 체크용.
   - **Light**  (T4 기준 약 1–2시간): 10×3 = 30 ep. 논문의 30-test 세팅과 비교 가능.
   - **Full**   (수 시간, Colab Pro+ 권장): 10×50 = 500 ep. Table I 정확 재현.
4. 태스크별 성공률 + multi-task 평균을 표로 출력.

> **런타임 요구사항**
> - 런타임 → 런타임 유형 변경 → GPU 선택 (T4 무료 / V100 · A100 Pro).
> - Colab의 headless 렌더링을 위해 `MUJOCO_GL=egl`이 필수입니다 (첫 셀에서 세팅).
> - 디스크 약 10 GB 필요 — LIBERO-10 hdf5(약 6 GB) + UVA 체크포인트(약 3 GB).

## 0. Python 3.10 설치

★ 처음 한 번만 실행. 자동 재시작 후 Python 버전 확인하고 다음 셀부터 진행.

In [ ]:
# 0. Python 3.10 설치 (condacolab)
# ★ 1회 자동 재시작 — "Python 3.10 OK" 뜨면 다음 셀로
import sys
print('Python:', sys.version)

if sys.version_info >= (3, 11):
    # conda install python=3.10 은 실행 중인 인터프리터를 바꿀 수 없어 실패함.
    # Miniconda py310 인스톨러로 환경 자체를 교체한 뒤 런타임을 재시작.
    print('Python >= 3.11 감지 → Miniconda Python 3.10 인스톨러로 교체 중...')
    !pip install -q condacolab
    import condacolab
    condacolab.install_from_url(
        "https://repo.anaconda.com/miniconda/Miniconda3-py310_23.5.2-0-Linux-x86_64.sh"
    )
    # ↑ 설치 완료 후 런타임 자동 재시작 → 이후 Python 3.10 으로 재진입
else:
    print('Python 3.10 OK → 아래 셀 계속 실행')

## 0. 런타임 체크
GPU 사용 가능 여부 확인, 그리고 robosuite·mujoco import 이전에 headless OpenGL 백엔드를 미리 설정합니다.

In [ ]:
import os, subprocess
os.environ["MUJOCO_GL"] = "egl"
os.environ["PYOPENGL_PLATFORM"] = "egl"
os.environ["MKL_THREADING_LAYER"] = "GNU"

print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)

: 

## 1. (선택) Google Drive 마운트 — 다운로드 영속화
다운로드 용량이 크기 때문에 Drive에 마운트해 두면 세션이 끊겨도 `checkpoints/`와 `data/`를 재사용할 수 있습니다.
`/content`에 임시로 두고 쓸 거라면 이 셀은 건너뛰세요.

In [ ]:
USE_DRIVE = False  # True로 바꾸면 /content/drive/MyDrive/uva_libero 에 계속 저장됨
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    PERSIST_ROOT = "/content/drive/MyDrive/uva_libero"
    os.makedirs(PERSIST_ROOT, exist_ok=True)
    print("영속화 루트:", PERSIST_ROOT)
else:
    PERSIST_ROOT = None
    print("Drive 미사용. 파일은 /content 아래에 저장됩니다.")

## 2. 시스템 패키지 설치 (apt)
Colab에서 `mujoco` headless 렌더링에 필요한 EGL + Mesa 라이브러리 설치.

In [ ]:
!apt-get update -qq
!apt-get install -y -qq \
    libegl1 libegl1-mesa libgles2-mesa \
    libgl1-mesa-glx libosmesa6 libosmesa6-dev \
    cmake libglfw3 libglew-dev patchelf ffmpeg \
    > /dev/null

## 3. 필요한 repo 클론
두 개의 repo가 **형제 위치**에 있어야 합니다:
- `unified_video_action/` — 이 논문의 원본 코드 (본인의 fork/branch 사용).
- `LIBERO/` — upstream LIBERO 벤치마크. UVA env runner가 `sys.path.append("../LIBERO")` 하므로 **반드시** `unified_video_action`의 형제 폴더여야 합니다.

In [ ]:
# ===== 아래 두 줄을 본인 fork/branch로 수정하세요 =====
UVA_REPO_URL    = "https://github.com/zser99/Unified-Video-Action-with-LLM.git"
UVA_REPO_BRANCH = "main"
# ===================================================

%cd /content
if not os.path.isdir("/content/unified_video_action"):
    !git clone --depth 1 -b {UVA_REPO_BRANCH} {UVA_REPO_URL} unified_video_action
if not os.path.isdir("/content/LIBERO"):
    !git clone --depth 1 https://github.com/Lifelong-Robot-Learning/LIBERO.git

!ls /content

## 4a. pip / setuptools 다운그레이드 (gym 0.21 필수 선행)
에러 `setup.py egg_info did not run successfully` / `metadata-generation-failed` 는 Colab의 **최신 pip·wheel** 이 `gym==0.21` 의 오래된 `setup.py` 를 거부해서 생깁니다.

**이 셀만 실행한 뒤 반드시 런타임 재시작** → 그다음 **4b** 셀을 실행하세요. (한 셀에 몰아서 설치하면 거의 항상 실패합니다.)

In [ ]:
import sys, subprocess
print("Python:", sys.version)
if sys.version_info >= (3, 12):
    print("Python 3.12+ → gym 0.21 불가. 4a 생략, 바로 4b 실행 (재시작 불필요).")
else:
    !apt-get install -y -qq python3-dev build-essential pkg-config \
        libavformat-dev libavcodec-dev libavdevice-dev libavutil-dev libswscale-dev libswresample-dev \
        > /dev/null
    !pip install -q "pip==21.3.1" "setuptools==65.5.0" "wheel==0.38.4"
    subprocess.run([sys.executable, "-m", "pip", "--version"], check=False)
    print("4a 완료 → Runtime 재시작 → 셀 0 → 4b")

### ⚠️ 여기서 반드시 재시작
**Runtime → Restart session** → **셀 0** (`MUJOCO_GL`) 다시 실행 → 아래 **4b** 셀 실행.

4a 없이 4b만 돌리면 `hydra` / `gym` 오류가 납니다.

## 4b. 나머지 패키지 설치 (재시작 **후** 실행)
4–7분 걸립니다. `dependency conflict` 경고는 대부분 무시해도 됩니다.

In [ ]:
import sys, subprocess, tempfile, os, tarfile

def _pip(*args, check=True):
    cmd = [sys.executable, "-m", "pip"] + list(args)
    print("$", " ".join(cmd))
    return subprocess.run(cmd, check=check)

def install_gym_colab():
    py = sys.version_info
    print(f"install_gym_colab: Python {py.major}.{py.minor}")
    if py >= (3, 12):
        _pip("install", "-q", "gym==0.26.2")
        import gym
        print("gym OK (0.26.2, Py>=3.12):", gym.__version__)
        return
    _pip("install", "-q", "pip==21.3.1", "setuptools==65.5.0", "wheel==0.38.4")
    r = _pip("install", "-q", "gym==0.21.0", check=False)
    if r.returncode == 0:
        import gym
        print("gym OK:", gym.__version__)
        return
    print("gym 0.21 failed — patch or fallback 0.26.2")
    td = tempfile.mkdtemp()
    r2 = _pip("download", "--no-deps", "-d", td, "gym==0.21.0", check=False)
    if r2.returncode != 0:
        _pip("install", "-q", "gym==0.26.2")
        import gym
        print("fallback gym OK:", gym.__version__)
        return
    tgz = [f for f in os.listdir(td) if f.endswith(".tar.gz")][0]
    root = os.path.join(td, "extract")
    os.makedirs(root, exist_ok=True)
    with tarfile.open(os.path.join(td, tgz)) as tf:
        tf.extractall(root)
    src = os.path.join(root, [d for d in os.listdir(root) if d.startswith("gym")][0])
    setup_py = os.path.join(src, "setup.py")
    text = open(setup_py, encoding="utf-8").read()
    text = text.replace("opencv-python>=3.", "opencv-python>=3.0")
    open(setup_py, "w", encoding="utf-8").write(text)
    _pip("install", "-q", src)
    import gym
    print("gym OK (patched):", gym.__version__)

install_gym_colab()

_pip("install", "-q",
     "numpy==1.26.4", "scipy==1.11.4", "numba==0.60.0",
     "hydra-core==1.2.0", "omegaconf==2.2.3",
     "dill==0.3.7", "einops==0.6.1",
     "diffusers==0.18.2", "transformers==4.28.0", "huggingface_hub==0.25.2",
     "timm==0.9.7", "accelerate==1.0.1",
     "zarr==2.16.1", "numcodecs==0.11.0", "imagecodecs==2024.12.30",
     "kornia==0.8.0", "kornia-rs==0.1.8",
     "opencv-python==4.11.0.86", "scikit-image==0.24.0",
     "pymunk==6.11.1", "pygame==2.6.1", "shapely==2.0.7",
     "h5py", "filelock", "threadpoolctl",
     "wandb==0.18.3", "gdown==5.2.0", "click", "pandas")

_pip("install", "-q", "av", check=False)
_pip("install", "-q", "mujoco==3.1.6", "robosuite==1.4.1")
_pip("install", "-q", "robomimic==0.3.0", check=False)
_pip("install", "-q", "-e", "/content/LIBERO")
_pip("install", "-q", "-e", "/content/unified_video_action", "--no-deps")

print("\n✅ 4b 완료 — 셀 5(체크포인트)부터 계속")

## 4c. (필요 시) diffusers 버전 고정

`cannot import name 'Union' from 'diffusers.optimization'` → **이 셀** 실행 후 import 검증·eval 재실행.

In [ ]:
import sys, subprocess
from pathlib import Path

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "diffusers==0.18.2", "transformers==4.28.0",
    "huggingface_hub==0.25.2", "accelerate==1.0.1",
])
import diffusers
print("diffusers", diffusers.__version__)

lr = Path("/content/unified_video_action/unified_video_action/model/common/lr_scheduler.py")
if lr.is_file() and "from diffusers.optimization import (\n    Union," in lr.read_text():
    lr.write_text(lr.read_text().replace(
        "from diffusers.optimization import (\n    Union,\n    SchedulerType,\n    Optional,\n    Optimizer,\n    TYPE_TO_SCHEDULER_FUNCTION,\n)",
        "from typing import Optional, Union\n\nfrom diffusers.optimization import (\n    SchedulerType,\n    Optimizer,\n    TYPE_TO_SCHEDULER_FUNCTION,\n)",
    ))
    print("patched lr_scheduler.py")
else:
    print("lr_scheduler OK")

## 5. UVA LIBERO-10 체크포인트 다운로드 (약 3 GB)

In [ ]:
import os
os.environ.setdefault("MUJOCO_GL", "egl")
%cd /content/unified_video_action
os.makedirs("checkpoints", exist_ok=True)

CKPT_PATH = "checkpoints/libero10.ckpt"
if not os.path.isfile(CKPT_PATH):
    !gdown 11c2VrmaRp48yw__5A5xpcu8EPzkexHSi -O {CKPT_PATH}
else:
    print("체크포인트가 이미 존재:", CKPT_PATH)

!ls -lh checkpoints

## 6. LIBERO-10 hdf5 데이터셋 다운로드
평가에는 **원본 hdf5 파일**(태스크당 1개)만 필요합니다. env runner가 `data/demo_*/states`로부터 초기 상태를 읽고, 언어 목표는 BDDL 파일명에서 직접 추출합니다. README의 CLIP 토큰화된 "변환된 데이터셋"은 *학습*용이므로 이 노트북에서는 불필요합니다.

In [ ]:
import os, glob
%cd /content/unified_video_action
os.makedirs("data/libero_10", exist_ok=True)

hdf5_files = glob.glob("data/libero_10/*.hdf5")
if len(hdf5_files) < 10:
    # 원본 LIBERO-10 hdf5 미러 (Google Drive, 압축 약 3 GB)
    !gdown 1_6Kc7e-s30MblbX8YjpxSofe9ZRPk3xv -O data/libero_10_raw.zip
    !unzip -oq data/libero_10_raw.zip -d data/
    # 압축이 보통 data/libero_10/ 폴더로 풀리지만, 다른 이름이면 정규화:
    for cand in ["libero10", "Libero10"]:
        p = f"data/{cand}"
        if os.path.isdir(p):
            !mv {p}/*.hdf5 data/libero_10/
    !rm -f data/libero_10_raw.zip

hdf5_files = sorted(glob.glob("data/libero_10/*.hdf5"))
print(f"탐지된 hdf5 파일 {len(hdf5_files)}개:")
for f in hdf5_files:
    print(" ", f)
assert len(hdf5_files) == 10, "LIBERO-10 태스크당 hdf5 10개를 예상했습니다. 위 다운로드 단계를 확인하세요."

## 7. import 검증
설치 오류를 미리 잡아 GPU 시간 낭비를 막습니다.

In [ ]:
import sys, os, importlib.util
os.environ.setdefault("MUJOCO_GL", "egl")
sys.path.insert(0, "/content/LIBERO")
sys.path.insert(0, "/content/unified_video_action")
sys.path.insert(0, "/content")

# 셀 4(의존성 설치) + 런타임 재시작을 건너뛰면 hydra 등이 없어서 여기서 실패합니다.
_missing = [m for m in ("hydra", "dill", "robomimic", "robosuite", "mujoco", "libero")
            if importlib.util.find_spec(m) is None]
if _missing:
    raise RuntimeError(
        f"패키지가 설치되지 않았습니다: {_missing}\n"
        "→ 위로 올라가 **셀 4(## 4. Python 의존성 설치)** 를 끝까지 실행하세요.\n"
        "→ **Runtime → Restart session** 후 **셀 0**부터 다시 실행하세요.\n"
        "→ 그다음 셀 5~7 순서로 진행하세요."
    )

import torch, hydra, dill
import robomimic, robosuite, mujoco
import libero
print("torch          :", torch.__version__, "| CUDA?", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "-")
print("mujoco         :", mujoco.__version__)
print("robosuite      :", robosuite.__version__)
print("robomimic      :", robomimic.__version__)
print("libero         :", libero.__file__)

from unified_video_action.workspace.base_workspace import BaseWorkspace
from unified_video_action.env_runner.base_image_runner import BaseImageRunner
print("UVA import OK")

## 8. 인라인 평가 함수 정의
`eval_sim.py`와 동일한 로직을 함수로 감싸서 셀마다 `n_test` 예산을 조절할 수 있게 합니다.

**중요**: `task_subset`은 `AsyncVectorEnv`가 생성되기 *전에* hdf5를 필터링합니다 — 그래서 스모크 테스트 한 번에 10 태스크분의 MuJoCo 워커가 떠는 사고를 방지합니다.

In [ ]:
import os, sys, json, glob, random, pathlib, dill, hydra, torch, numpy as np
from omegaconf import open_dict

%cd /content/unified_video_action
sys.path.insert(0, "/content/unified_video_action")

from unified_video_action.workspace.base_workspace import BaseWorkspace
from unified_video_action.env_runner.base_image_runner import BaseImageRunner


def _instantiate_libero_runners(cfg, output_dir, task_subset):
    """utils.load_env.load_env_runner 와 동일하지만, AsyncVectorEnv가
    생성되기 전에 task_subset을 적용해 10개 태스크분의 자원을 아낀."""
    hdf5_files = sorted(glob.glob(cfg.task.dataset.dataset_path + "/*hdf5"))
    print(f"Found {len(hdf5_files)} HDF5 files in {cfg.task.dataset.dataset_path}:")
    for f in hdf5_files:
        print("  ", os.path.basename(f))
    if task_subset:
        hdf5_files = [f for f in hdf5_files
                      if any(s.lower() in os.path.basename(f).lower() for s in task_subset)]
        assert hdf5_files, f"task_subset {task_subset} matched nothing in above list"
    print(f"env_runner 생성 — 태스크 {len(hdf5_files)}개:")
    runners = []
    for f in hdf5_files:
        runner = hydra.utils.instantiate(cfg.task.env_runner, task_dir=f, output_dir=output_dir)
        assert isinstance(runner, BaseImageRunner)
        runners.append(runner)
    return runners


def run_libero10_eval(checkpoint="checkpoints/libero10.ckpt",
                      output_dir="outputs/libero10_eval",
                      device="cuda:0",
                      n_test=3,
                      n_train=0,
                      n_envs=1,
                      n_test_vis=1,
                      max_steps=None,
                      task_subset=None,
                      debug=False):
    """함수 인수 설명:
        n_test     : 태스크당 rollout 수 (논문=50). 스모크는 1–3, light는 3–5, full은 50.
        n_train    : train rollout 수 (0으로 두면 건너뜀 — pre-collected action 불필요).
        n_envs     : chunk당 병렬 env 수. T4에서는 1이 가장 안정적.
        n_test_vis : mp4로 저장할 rollout 수 (0이면 미저장).
        task_subset: 부분 문자열 리스트. hdf5 basename에 하나라도 포함되면
                     해당 태스크만 평가. env 생성 전에 필터링됨.
        max_steps  : 에피소드당 최대 스텝 수 (cfg 기본값=500).
    """
    pathlib.Path(output_dir).mkdir(parents=True, exist_ok=True)

    payload = torch.load(open(checkpoint, "rb"), pickle_module=dill, map_location="cpu")
    cfg = payload["cfg"]

    seed = cfg.training.seed
    torch.manual_seed(seed); np.random.seed(seed); random.seed(seed)

    with open_dict(cfg):
        cfg.output_dir = output_dir
        cfg.task.env_runner.n_test = int(n_test)
        cfg.task.env_runner.n_train = int(n_train)
        cfg.task.env_runner.n_envs = int(n_envs)
        cfg.task.env_runner.n_test_vis = min(n_test_vis, n_test)
        cfg.task.env_runner.n_train_vis = 0
        if max_steps is not None:
            cfg.task.env_runner.max_steps = int(max_steps)
        cfg.training.debug = bool(debug)

    cls = hydra.utils.get_class(cfg.model._target_)
    workspace = cls(cfg, output_dir=output_dir)
    workspace.load_payload(payload, exclude_keys=None, include_keys=None)

    policy = workspace.ema_model
    policy.to(device)
    policy.eval()

    env_runners = _instantiate_libero_runners(cfg, output_dir, task_subset)

    step_log = {}
    for env_runner in env_runners:
        runner_log = env_runner.run(policy)
        step_log.update(runner_log)
        print(f"[{env_runner.task_name}] keys=",
              [k for k in runner_log if 'mean_score' in k])
        try:
            env_runner.env.close()
        except Exception:
            pass

    test_scores = {k: v for k, v in step_log.items()
                   if 'test/' in k and '_mean_score' in k}
    step_log["test_mean_score"] = (
        float(np.mean(list(test_scores.values()))) if test_scores else float("nan")
    )

    import wandb
    json_log = {}
    for k, v in step_log.items():
        if isinstance(v, wandb.sdk.data_types.video.Video):
            json_log[k] = v._path
        else:
            try:
                json.dumps(v)
                json_log[k] = v
            except TypeError:
                json_log[k] = str(v)
    out_path = os.path.join(output_dir,
                            f"eval_log_{os.path.basename(checkpoint)}.json")
    with open(out_path, "w") as f:
        json.dump(json_log, f, indent=2, sort_keys=True)
    print("저장됨:", out_path)

    return step_log, json_log

## 9. Smoke 테스트 (T4 약 10–20분)
LIBERO-10 태스크 중 하나만 골라 **1 rollout**만 돌려서 파이프라인 자체가 살아있는지 확인합니다.

In [ ]:
step_log_smoke, json_smoke = run_libero10_eval(
    output_dir="outputs/libero10_smoke",
    n_test=1,
    n_train=0,
    n_envs=1,
    n_test_vis=1,
    max_steps=300,                # 피드백을 빨리 받기 위해 짧은 horizon
    task_subset=["moka_pot"],    # 한 태스크만; 제거하면 10개 전체
)
print("smoke test_mean_score:", step_log_smoke.get("test_mean_score"))

## 10. Light 평가 (T4 약 1–2시간)
10 태스크 × 3 rollout = 30 ep. 논문 Supplementary Table VIII의 "30-test" 세팅과 동일한 조건이며, 논문 수치는 **UVA = 0.93**입니다.

In [ ]:
step_log_light, json_light = run_libero10_eval(
    output_dir="outputs/libero10_light",
    n_test=3,
    n_train=0,
    n_envs=1,
    n_test_vis=0,                # 디스크 절약을 위해 별도 영상 저장 안 함
)
print("light test_mean_score:", step_log_light.get("test_mean_score"))

## 11. Full 재현 (Colab Pro+ 권장)
10 태스크 × 50 rollout = 500 ep. 논문 Table I (`UVA = 0.90`)을 그대로 재현합니다. 단일 A100 기준 약 6–12시간 소요이므로 무료 Colab 세션으로는 구동이 어렵습니다. Pro+를 쓰거나, `task_subset=[...]`로 여러 번에 나눠 돌리세요.

In [ ]:
# 장시간 런타임이 준비되면 주석을 해제하세요
# step_log_full, json_full = run_libero10_eval(
#     output_dir="outputs/libero10_full",
#     n_test=50,
#     n_train=0,
#     n_envs=2,
#     n_test_vis=0,
# )
# print("full test_mean_score:", step_log_full.get("test_mean_score"))

## 12. 결과 파싱 · 논문 수치와 비교

In [ ]:
import pandas as pd

def summarize(step_log, label):
    rows = []
    for k, v in step_log.items():
        if 'test/' in k and '_mean_score' in k:
            task = k.replace('test/', '').replace('_mean_score', '')
            rows.append({'task': task, 'success_rate': float(v)})
    if not rows:
        print(f"[{label}] 태스크별 점수가 없습니다")
        return None
    df = pd.DataFrame(rows).sort_values('task').reset_index(drop=True)
    df.loc[len(df)] = ['_AVERAGE_', df['success_rate'].mean()]
    print(f"\n=== {label} ===")
    print(df.to_string(index=False))
    return df

for log, lbl in [
    (locals().get('step_log_smoke'), 'smoke (1 ep/task)'),
    (locals().get('step_log_light'), 'light (3 ep/task)'),
    (locals().get('step_log_full'),  'full  (50 ep/task)'),
]:
    if log is not None:
        summarize(log, lbl)

print("\n논문 수치 (Table I)         : UVA = 0.90 on LIBERO-10, 태스크당 50 rollout.")
print("논문 수치 (Suppl. Table VIII): UVA = 0.93, 30-test 세팅 (태스크당 3 rollout).")

## 트러블슈팅 팁
- **`Union` from `diffusers.optimization`** → **4c 셀**(`colab_libero10_cot_eval.ipynb`와 동일: `diffusers==0.18.2` 고정) 실행 후 import 검증·eval 재실행.
- **`mujoco.FatalError: gladLoadGL error`** → robosuite/mujoco import 이전에 `MUJOCO_GL=egl`이 설정되지 않은 경우. 런타임을 재시작하고 셀 0부터 다시 실행하세요.
- **`AsyncVectorEnv`가 `env.reset()`에서 멈춤** → `n_envs=1`로 낮추세요. Colab에서는 다중 프로세스 MuJoCo가 불안정합니다.
- **`bddl_file ... not in bddl_file_name_dict.values()`** → `unified_video_action/env/libero/bddl_files/libero_10/*.bddl` 파일이 존재해야 합니다. repo에 vendored되어 있으므로 shallow clone도 충분합니다.
- **액션 디퓨전 중 CUDA OOM** → 체크포인트 로드 후 `cfg.model.policy.autoregressive_model_params.num_iter`를 줄이거나 diffusion step 수를 낮추세요.
- **UVA + CoT/LLM 평가** → 같은 repo의 `notebooks/colab_libero10_cot_eval.ipynb` 를 사용하세요 (baseline vs rule vs LLM 비교).